# Seed media — 115 photographs for brgen, amber and bsdports

**Runtime → Change runtime type → T4 GPU** before running.

Set `HF_TOKEN` in the sidebar (🔑). SDXL is not licence-gated and needs no
token, but the setup cell keeps the check so a later `SEED_MEDIA_BASE=flux`
run fails on a missing licence here rather than four hundred lines into a
download.

SDXL for all of them, and that is the hardware rather than a preference.

FLUX.1-dev cannot be **loaded** on a free Colab, never mind fitted in VRAM.
Its shards are 23.8 GB of fp16 and diffusers materialises them in host RAM
before quantisation can move anything to the card; this box has 12.7 GB and
the kernel dies at `Loading checkpoint shards`. nf4 does not help — the model
has to exist before it can be made smaller. `colab_session.rb:136` records
the same two deaths from the training lane.

1. **dating** — 32 frames, general-model strangers, no adapter
2. **scenes** — 63 frames
3. **amber** — 20 frames, each on the one backdrop

`SEED_MEDIA_BASE=flux` switches to FLUX on a high-RAM runtime, where the
load actually fits. On free Colab it will be killed.

Roughly 20–40 s a frame on a T4 at nf4, so about half an hour of GPU plus the
model download. Output goes to Drive; a disconnect costs the frames since the
last write and not the session.

Prompts are `STUDIO/lora/seed_media.yml`. Edits belong there — this notebook is
generated by `_toolkit/run_seed_media_colab.rb` and is overwritten.


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("ok: HF_TOKEN from Colab secrets")
except Exception:
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (FLUX.1-dev licence accepted): ")
assert os.environ.get("HF_TOKEN"), "no HF token"

# Checked, not enforced -- unless FLUX is actually the base.
#
# SDXL is not licence-gated and needs no token, so refusing the run over a
# FLUX licence would block the default path on the terms of a model it does
# not use. That is check_hf_flux_access.rb's own recorded bug, which blocked
# SDXL renders on machines with no HF credentials at all.
#
# It still fails hard when SEED_MEDIA_BASE=flux, because there the 403 arrives
# hundreds of lines later inside diffusers, named as a missing file.
import urllib.request
req = urllib.request.Request(
    "https://huggingface.co/api/models/black-forest-labs/FLUX.1-dev",
    headers={"Authorization": f"Bearer {os.environ['HF_TOKEN']}"})
try:
    urllib.request.urlopen(req).read(1)
    print("ok: FLUX.1-dev reachable with this token")
except Exception as e:
    msg = ("FLUX.1-dev is not reachable with this token. Accept the licence at "
           "https://huggingface.co/black-forest-labs/FLUX.1-dev")
    if os.environ.get("SEED_MEDIA_BASE", "sdxl").lower() == "flux":
        raise SystemExit(msg + " — required because SEED_MEDIA_BASE=flux. " + str(e))
    print("note:", msg)
    print("note: not needed for the SDXL default; continuing.")


In [ ]:
# Frames go to Drive as they are made. A free session disconnects when idle and
# is capped near 12 h; writing at the end would make every disconnect cost the
# whole run.
import os
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/seed_media"
os.makedirs(OUT, exist_ok=True)
print("ok:", OUT)


In [ ]:
import subprocess, os
# bitsandbytes for nf4, sentencepiece for T5's tokenizer.
subprocess.run(
    "pip -q install -U diffusers transformers accelerate safetensors "
    "bitsandbytes sentencepiece protobuf",
    shell=True, check=True)

if not os.path.isdir("/content/pub4/.git"):
    subprocess.run(
        "git clone --depth 1 --branch main https://github.com/anon987654321/pub4.git /content/pub4",
        shell=True, check=True)
print("ok: toolkit and prompts at /content/pub4")

# Assert the GPU before anything expensive believes it has one.
#
# Without this the notebook installs, clones, downloads twenty gigabytes and
# then renders on the CPU, where a 12B transformer is minutes per step -- so
# the first sign of a missed Runtime -> T4 is a frame that never arrives.
# Colab also nags about an idle GPU while these setup cells run, which is
# correct and means nothing; this is the check that does mean something.
import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "no CUDA device. Runtime -> Change runtime type -> T4 GPU, then rerun. "
        "Do not switch to a standard runtime: FLUX.1-dev on CPU is minutes per step.")
free, total = torch.cuda.mem_get_info()
print(f"ok: {torch.cuda.get_device_name(0)}, {total / 1e9:.1f} GB total, {free / 1e9:.1f} GB free")
if total < 14e9:
    print("warn: under 14 GB — nf4 fits a 16 GB T4 with little to spare")


In [ ]:
import gc, glob, json, os, yaml, torch

SPEC = "/content/pub4/STUDIO/lora/seed_media.yml"
spec = yaml.safe_load(open(SPEC))
DTYPE = torch.float16  # Turing has no bf16.

# SDXL for all sixty-one, and this is the same conclusion the training lane
# reached rather than a new opinion.
#
# FLUX.1-dev cannot be LOADED here, never mind fitted in VRAM. Its shards are
# 23.8 GB of fp16 and diffusers materialises them in host RAM before
# quantisation can move anything to the card; a free Colab has 12.7 GB and the
# kernel is killed at "Loading checkpoint shards". nf4 does not help, because
# the model has to exist before it can be made smaller. colab_session.rb line
# 136 records the same two deaths.
#
# add_swap puts 24 GB on the scratch disk and lets the loader spill, which
# sometimes works and sometimes meets a container that refuses swapon. So FLUX
# is opt-in rather than the default: SEED_MEDIA_BASE=flux on a high-RAM
# runtime. The default is the one that produces photographs tonight.
BASE = os.environ.get("SEED_MEDIA_BASE", "sdxl").lower()
print(f"ok: base = {BASE}")

RATIOS = {"3:2": (1216, 832), "4:5": (896, 1120), "1:1": (1024, 1024), "4:3": (1152, 864)}

def size_for(ratio):
    return RATIOS.get(ratio or spec["meta"]["aspect_ratio"], (1024, 1024))

def render(pipe, key, prompt, ratio, seed, **kw):
    path = os.path.join(OUT, key + ".png")
    if os.path.exists(path):          # Resume is free; a disconnect is not.
        print("skip", key); return
    w, h = size_for(ratio)
    img = pipe(prompt=prompt, width=w, height=h,
               generator=torch.Generator("cpu").manual_seed(seed), **kw).images[0]
    img.save(path)
    print("ok", key, f"{w}x{h}")


# --------------------------------------------------------------- the model
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
# The fp16-fix VAE, not SDXL's own: the original overflows in fp16 and decodes
# to black. render_config.rb names the same repo for the same reason.
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=DTYPE)
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae, torch_dtype=DTYPE, variant="fp16", use_safetensors=True)
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)
KW = dict(guidance_scale=6.0, num_inference_steps=30)
print("ok: SDXL loaded")

# ------------------------------------------------------------------ pass 1
# No subject adapter: the seeded people are general-model strangers, because
# the seeds publish on public hosts.
dating = spec["dating"]
people = {**dating["profiles"], **(dating.get("pool") or {})}
for i, (key, prompt) in enumerate(people.items()):
    render(pipe, key, prompt, dating["aspect_ratio"], 1000 + i, **KW)

# ------------------------------------------------------------ passes 2 & 3
for i, (key, entry) in enumerate(spec["scenes"].items()):
    render(pipe, key, entry["prompt"], entry.get("aspect_ratio"), 2000 + i, **KW)

# The backdrop clause is appended rather than repeated per entry, so every
# garment shares one backdrop instead of seventeen near-misses.
amber = spec["amber"]
garments = {**amber["garments"], **(amber.get("outfits") or {})}
for i, (key, garment) in enumerate(garments.items()):
    render(pipe, key, garment + ", " + spec["backdrop"], amber["aspect_ratio"], 3000 + i, **KW)

made = len([f for f in os.listdir(OUT) if f.endswith(".png")])
print("")
print(f"ok: {made} frame(s) in {OUT}")
print("next: download that folder, then on the Mac:")
print("  ruby STUDIO/lora/_toolkit/install_seed_media.rb <folder>")
